# 🔧 題目 1：零售 POS 銷售分析 — Solution
# Mini Data Pipeline 工作坊

> **情境**：你是一家零售連鎖集團的資料顧問。老闆想知道哪些商品最暢銷、哪些客戶最有價值、各國市場表現如何。
>
> **Pipeline**：`CSV → pandas → SQLite (raw/cleaned/analyzed) → SQL → LLM → FastAPI → Streamlit`
>
> **資料**：[Kaggle: Online Retail II UCI](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci)（2,000 筆取樣）

⚠️ **這是完整 Solution 版，僅供講師參考。學員請用 `pipeline_starter.ipynb`。**


## Section 0：環境設定


In [ ]:
import pandas as pd
import sqlite3
import os
import json

print("✅ 套件載入完成")


In [ ]:
# API Key 設定
OPENAI_API_KEY = ""  # 講師提供共用 Key

if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]

print("✅ API Key 已設定" if OPENAI_API_KEY else "⚠️ 無 API Key，使用 fallback")


---
## Section 1：Extract — 讀取資料 + 寫入 raw 表


In [ ]:
# 讀取 CSV
df_raw = pd.read_csv("data/raw/topic_1/orders.csv")
print(f"📊 {len(df_raw)} 筆, {len(df_raw.columns)} 欄")
print(f"欄位: {list(df_raw.columns)}")
df_raw.head()


In [ ]:
# 檢查資料品質
print("=== 型別 ===")
print(df_raw.dtypes)
print("\n=== 缺漏值 ===")
print(df_raw.isnull().sum())
print("\n=== 數值統計 ===")
print(df_raw.describe())


In [ ]:
# 建立 SQLite + 寫入 raw 表
DB_PATH = "pipeline.db"
conn = sqlite3.connect(DB_PATH)
df_raw.to_sql("raw_orders", conn, if_exists="replace", index=False)

result = pd.read_sql("SELECT COUNT(*) as total FROM raw_orders", conn)
print(f"✅ raw_orders: {result['total'][0]} 筆")


---
## Section 2：Transform — 清洗 + 寫入 cleaned 表


In [ ]:
# 從 raw 表讀出
df = pd.read_sql("SELECT * FROM raw_orders", conn)
before = len(df)

# 1. 處理缺漏值
df = df.dropna(subset=["description", "customer_id"])

# 2. 日期轉換
df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df["year"] = df["invoice_date"].dt.year
df["month"] = df["invoice_date"].dt.month
df["day_of_week"] = df["invoice_date"].dt.day_name()
df["hour"] = df["invoice_date"].dt.hour

# 3. 確保數值正確
df["total_amount"] = df["quantity"] * df["unit_price"]

# 4. 過濾異常值
df = df[(df["quantity"] > 0) & (df["unit_price"] > 0)]

print(f"清洗前: {before} → 清洗後: {len(df)} 筆")
print(f"缺漏值: {df.isnull().sum().sum()}")


In [ ]:
# 檢查點
assert df.isnull().sum().sum() == 0, "❌ 還有缺漏值"
assert (df["quantity"] > 0).all(), "❌ quantity 有負值"
assert (df["unit_price"] > 0).all(), "❌ unit_price 有負值"
print("✅ 全部檢查通過")


In [ ]:
# 寫入 cleaned 表
df.to_sql("cleaned_orders", conn, if_exists="replace", index=False)
print(f"✅ cleaned_orders: {len(df)} 筆")


---
## Section 3：SQL 統計分析


In [ ]:
# 商品銷售排行
product_stats = pd.read_sql("""
SELECT description, 
       COUNT(*) as order_count,
       SUM(quantity) as total_qty,
       ROUND(SUM(total_amount), 2) as total_revenue
FROM cleaned_orders
GROUP BY description
ORDER BY total_revenue DESC
LIMIT 20
""", conn)
print("📊 商品銷售 Top 20：")
product_stats


In [ ]:
# 各國銷售統計
country_stats = pd.read_sql("""
SELECT country,
       COUNT(DISTINCT customer_id) as customers,
       COUNT(*) as orders,
       ROUND(SUM(total_amount), 2) as revenue
FROM cleaned_orders
GROUP BY country
ORDER BY revenue DESC
""", conn)
print("📊 各國銷售：")
country_stats


In [ ]:
# 視覺化
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
product_stats.head(10).plot.barh(x="description", y="total_revenue", ax=axes[0], color="steelblue")
axes[0].set_title("商品銷售額 Top 10")
country_stats.head(10).plot.barh(x="country", y="revenue", ax=axes[1], color="coral")
axes[1].set_title("各國銷售額 Top 10")
plt.tight_layout()
plt.show()


In [ ]:
# 存統計結果
os.makedirs("data/processed", exist_ok=True)
product_stats.to_csv("data/processed/product_stats.csv", index=False)
country_stats.to_csv("data/processed/country_stats.csv", index=False)
print("✅ 統計結果已存到 data/processed/")


---
## Section 4：LLM 加值分析 + 寫入 analyzed 表


In [ ]:
import requests

def llm_analyze(text, api_key=None):
    if api_key:
        return _llm_api(text, api_key)
    return _llm_fallback(text)

def _llm_api(text, api_key):
    prompt = f"""請分析以下零售商品描述，回傳 JSON：
{{"category": "家飾/禮品/餐具/季節商品/文具/其他", "insight": "一句話商品洞察"}}

商品描述：{text[:300]}"""
    try:
        resp = requests.post("https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3},
            timeout=30)
        content = resp.json()["choices"][0]["message"]["content"].strip()
        if content.startswith("```"): content = content.split("\n", 1)[1].rsplit("```", 1)[0]
        return json.loads(content)
    except:
        return _llm_fallback(text)

def _llm_fallback(text):
    t = text.lower()
    if any(w in t for w in ["christmas", "xmas", "santa", "snowman", "winter"]):
        cat = "季節商品"
    elif any(w in t for w in ["candle", "holder", "frame", "clock", "lamp", "mirror"]):
        cat = "家飾"
    elif any(w in t for w in ["cup", "mug", "plate", "bowl", "jar", "bottle"]):
        cat = "餐具"
    elif any(w in t for w in ["pen", "pencil", "notebook", "card", "sticker"]):
        cat = "文具"
    elif any(w in t for w in ["gift", "bag", "box", "ribbon", "wrap"]):
        cat = "禮品"
    else:
        cat = "其他"
    return {"category": cat, "insight": text[:50] + "..."}

print("✅ LLM Helper 已定義")


In [ ]:
# 單筆測試
test = df["description"].iloc[0]
print(f"📝 輸入: {test}")
result = llm_analyze(test, OPENAI_API_KEY if OPENAI_API_KEY else None)
print(f"🤖 分類: {result['category']}, 洞察: {result['insight']}")


In [ ]:
# 批次分析
BATCH_SIZE = 50
api_key = OPENAI_API_KEY if OPENAI_API_KEY else None
print(f"模式: {'API' if api_key else 'Fallback'}, 分析 {BATCH_SIZE} 筆...")

results = []
for i, row in df.head(BATCH_SIZE).iterrows():
    r = llm_analyze(str(row["description"]), api_key)
    results.append(r)
    if len(results) % 10 == 0: print(f"  進度: {len(results)}/{BATCH_SIZE}")

df_analyzed = df.head(BATCH_SIZE).copy()
df_analyzed["category"] = [r["category"] for r in results]
df_analyzed["llm_insight"] = [r["insight"] for r in results]

print(f"\n✅ 完成 {len(results)} 筆")
print(f"\n📊 品類分佈：\n{df_analyzed['category'].value_counts()}")


In [ ]:
# 寫入 analyzed 表
df_analyzed.to_sql("analyzed_orders", conn, if_exists="replace", index=False)
print("\n📊 三表狀態：")
for t in ["raw_orders", "cleaned_orders", "analyzed_orders"]:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn)["n"][0]
    print(f"  {t}: {n} 筆")


---
## Section 5：驗證 pipeline


In [ ]:
# 跨表查詢
lineage = pd.read_sql("""
SELECT 'raw_orders' as layer, COUNT(*) as rows FROM raw_orders
UNION ALL SELECT 'cleaned_orders', COUNT(*) FROM cleaned_orders
UNION ALL SELECT 'analyzed_orders', COUNT(*) FROM analyzed_orders
""", conn)
print("📊 Pipeline 資料流：")
print(lineage.to_string(index=False))


In [ ]:
# 品類 × 銷售額交叉
cross = pd.read_sql("""
SELECT category, COUNT(*) as items, ROUND(SUM(total_amount), 2) as revenue
FROM analyzed_orders
GROUP BY category ORDER BY revenue DESC
""", conn)
print("📊 品類 × 銷售額：")
cross


---
## Section 6：產出報告


In [ ]:
# 收集數據
total_rev = pd.read_sql("SELECT ROUND(SUM(total_amount),2) as r FROM cleaned_orders", conn)["r"][0]
top_products = product_stats.head(5).to_dict("records")
top_countries = country_stats.head(5).to_dict("records")
cat_dist = df_analyzed["category"].value_counts().to_dict()

report = f"""# 零售 POS 銷售分析報告

## 資料概要
- 分析筆數：{len(df)} 筆交易
- 總銷售額：${total_rev:,.2f}
- 資料來源：UCI Online Retail II

## 商品銷售 Top 5
{chr(10).join(f'- {p["description"]}: ${p["total_revenue"]:,.2f}' for p in top_products)}

## 各國銷售 Top 5
{chr(10).join(f'- {c["country"]}: ${c["revenue"]:,.2f} ({c["customers"]} 位客戶)' for c in top_countries)}

## 品類分佈（LLM 分析 {len(df_analyzed)} 筆）
{chr(10).join(f'- {cat}: {cnt} 筆' for cat, cnt in cat_dist.items())}

## 建議
1. 季節商品佔比高，建議提前備貨
2. 優先維護高消費客戶
3. 定期追蹤各國銷售趨勢

## Pipeline
CSV → pandas → SQLite(raw/cleaned/analyzed) → SQL → LLM → 本報告
"""

os.makedirs("output", exist_ok=True)
with open("output/report.md", "w") as f: f.write(report)
print("✅ 報告已存到 output/report.md")
print(report)


---
## Section 7：打包確認


In [ ]:
checks = [
    ("pipeline.db", "SQLite 資料庫"),
    ("data/processed/product_stats.csv", "商品統計"),
    ("data/processed/country_stats.csv", "國家統計"),
    ("output/report.md", "顧問報告"),
]
print("📋 產出確認：")
for path, desc in checks:
    print(f"  {'✅' if os.path.exists(path) else '❌'} {desc}: {path}")

if os.path.exists("pipeline.db"):
    c = sqlite3.connect("pipeline.db")
    for t in ["raw_orders", "cleaned_orders", "analyzed_orders"]:
        try:
            n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", c)["n"][0]
            print(f"  ✅ SQLite 表 {t}: {n} 筆")
        except: print(f"  ❌ SQLite 表 {t} 不存在")
    c.close()

print("\n📋 接下來：填 README + upgrade_plan + 準備 3 分鐘 Demo")


---
## Section 8：FastAPI — 在 Notebook 裡啟動 API


In [ ]:
# 安裝 + import
!pip install -q fastapi uvicorn nest_asyncio
from fastapi import FastAPI
import nest_asyncio
nest_asyncio.apply()

api = FastAPI(title="零售銷售分析 API")

@api.get("/health")
def health():
    return {"status": "ok"}

@api.get("/stats")
def get_stats():
    c = sqlite3.connect("pipeline.db")
    df = pd.read_sql("""
        SELECT description, COUNT(*) as orders, ROUND(SUM(total_amount),2) as revenue
        FROM cleaned_orders GROUP BY description ORDER BY revenue DESC LIMIT 20
    """, c)
    c.close()
    return df.to_dict(orient="records")

@api.get("/analyzed")
def get_analyzed():
    c = sqlite3.connect("pipeline.db")
    df = pd.read_sql("SELECT description, category, llm_insight FROM analyzed_orders LIMIT 20", c)
    c.close()
    return df.to_dict(orient="records")

print("✅ API 定義完成（3 個 endpoint）")


In [ ]:
# 啟動 + 測試
import threading, uvicorn, time, requests

thread = threading.Thread(target=uvicorn.run, args=(api,), kwargs={"host": "0.0.0.0", "port": 8000, "log_level": "warning"})
thread.daemon = True
thread.start()
time.sleep(2)

print("📡 /health:", requests.get("http://localhost:8000/health").json())
print("📡 /stats (前 3 筆):", requests.get("http://localhost:8000/stats").json()[:3])
print("📡 /analyzed (前 3 筆):", requests.get("http://localhost:8000/analyzed").json()[:3])


完整版在 `api.py`，本地跑 `uvicorn api:app --reload --port 8000`。


---
## Section 9：Dashboard — ipywidgets 互動圖表


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

df_dash = pd.read_sql("SELECT * FROM cleaned_orders", conn)

country_dropdown = widgets.Dropdown(
    options=["全部"] + sorted(df_dash["country"].unique().tolist()),
    description="選國家："
)

def update_dashboard(country):
    clear_output(wait=True)
    display(country_dropdown)
    data = df_dash if country == "全部" else df_dash[df_dash["country"] == country]
    
    print(f"📊 {country}: {len(data)} 筆, 營收 ${data['total_amount'].sum():,.2f}")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    data.groupby("description")["total_amount"].sum().sort_values().tail(10).plot.barh(ax=axes[0], color="steelblue")
    axes[0].set_title(f"商品銷售額 Top 10 — {country}")
    
    if "hour" in data.columns:
        data.groupby("hour")["total_amount"].sum().plot(ax=axes[1], color="coral", marker="o")
        axes[1].set_title(f"每小時銷售額 — {country}")
    
    plt.tight_layout()
    plt.show()

widgets.interact(update_dashboard, country=country_dropdown)


完整版在 `app.py`，本地跑 `streamlit run app.py`。


---
## Section 10：本地部署指引

```bash
# 環境準備
pip install fastapi uvicorn streamlit pandas

# 啟動 API
cd data/raw/topic_1
uvicorn api:app --reload --port 8000
# 開 http://localhost:8000/docs 看 Swagger UI

# 另開 terminal，啟動 Dashboard  
streamlit run app.py
```

完整架構：`Notebook → pipeline.db → api.py（API）→ app.py（Dashboard）`
